# Diabetes Model — Hyperparameter Tuning

This notebook performs hyperparameter tuning for the baseline diabetes
classification models developed in Healytics.

### Objectives

- Recreate the training and testing datasets
- Preserve the same preprocessing strategy
- Use cross-validation on the training data
- Tune selected model hyperparameters
- Compare tuned models with their baseline versions

### Important Evaluation Rule

The test set is not used during hyperparameter selection.

Cross-validation is performed only on the training data. The test set remains
untouched until the final evaluation stage.

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier

## 2. Load and Prepare the Diabetes Dataset

The same raw dataset and preprocessing decisions used in previous stages are
recreated.

Invalid zero-coded measurements are converted to missing values.

The same random state and stratified 80/20 split are used to maintain
consistency with the previous evaluation.

In [2]:
df = pd.read_csv("../data/raw/diabetes.csv")

invalid_zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

df[invalid_zero_columns] = df[invalid_zero_columns].replace(0, np.nan)

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (614, 8)
Testing set: (154, 8)


## 3. Random Forest Preprocessing and Model Pipeline

A reusable pipeline is created containing:

1. Median imputation
2. Standardization
3. Random Forest classifier

The preprocessing is included inside the pipeline so that each cross-validation
fold learns its preprocessing parameters only from its corresponding training
portion.

In [3]:
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(
        random_state=42
    ))
])

## 4. Random Forest Hyperparameter Grid

Hyperparameters control how the Random Forest model is constructed.

The initial search considers:

- `n_estimators` — number of trees
- `max_depth` — maximum depth of each tree
- `min_samples_split` — minimum samples required to split an internal node
- `min_samples_leaf` — minimum samples required at a leaf node

A limited grid is used initially to keep the search computationally manageable.

In [4]:
rf_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

## 5. Hyperparameter Search

GridSearchCV evaluates every parameter combination using 5-fold
cross-validation on the training dataset.

ROC-AUC is used as the optimization metric because it evaluates the model's
ability to distinguish between the two classes across different classification
thresholds.

The test dataset is not used during this search.

In [5]:
rf_grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    refit=True
)

In [6]:
rf_grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [None, 5, ...], 'model__min_samples_leaf': [1, 2], 'model__min_samples_split': [2, 5], 'model__n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callab

In [8]:
print("Random Forest tuning completed.")

Random Forest tuning completed.


## 6. Best Random Forest Parameters

The parameter combination with the highest mean cross-validated ROC-AUC on
the training data is retrieved.

The test set has not been used to select these parameters.

In [9]:
print("Best parameters:")
print(rf_grid_search.best_params_)

Best parameters:
{'model__max_depth': 5, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 100}


In [10]:
print("Best cross-validation ROC-AUC:")
print(round(rf_grid_search.best_score_, 4))

Best cross-validation ROC-AUC:
0.8375


In [12]:
best_rf_model = rf_grid_search.best_estimator_

print("Best Random Forest pipeline created successfully.")

Best Random Forest pipeline created successfully.


## 7. Gradient Boosting Hyperparameter Tuning

Gradient Boosting was included because its baseline model achieved a ROC-AUC
of approximately 0.8302 on the initial test split.

Hyperparameter tuning is performed using 5-fold cross-validation on the
training dataset only.

The test dataset remains untouched during parameter selection.

In [13]:
from sklearn.ensemble import GradientBoostingClassifier

### Gradient Boosting Pipeline

The pipeline combines median imputation, standardization, and the Gradient
Boosting classifier.

In [14]:
gb_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", GradientBoostingClassifier(
        random_state=42
    ))
])

### Hyperparameter Grid

The search considers the number of boosting stages, learning rate, maximum
tree depth, and minimum samples required to split a node.

In [15]:
gb_param_grid = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [2, 3],
    "model__min_samples_split": [2, 5]
}

In [16]:
gb_grid_search = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=gb_param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    refit=True
)

In [17]:
gb_grid_search.fit(X_train, y_train)

print("Gradient Boosting tuning completed.")

Gradient Boosting tuning completed.


In [20]:
print("Best parameters:")
print(gb_grid_search.best_params_)

print("\nBest cross-validation ROC-AUC:")
print(round(gb_grid_search.best_score_, 4))

best_gb_model = gb_grid_search.best_estimator_
print("\nBest Gradient Boosting pipeline created successfully.")

Best parameters:
{'model__learning_rate': 0.05, 'model__max_depth': 2, 'model__min_samples_split': 2, 'model__n_estimators': 100}

Best cross-validation ROC-AUC:
0.8297

Best Gradient Boosting pipeline created successfully.


## 8. Logistic Regression Hyperparameter Tuning

Logistic Regression is tuned using 5-fold cross-validation on the training
dataset.

The search focuses on the regularization strength (`C`) and the regularization
type (`penalty`).

In [21]:
from sklearn.linear_model import LogisticRegression

In [22]:
lr_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

In [23]:
lr_param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l2"]
}

In [24]:
lr_grid_search = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    refit=True
)

In [25]:
lr_grid_search.fit(X_train, y_train)

print("Logistic Regression tuning completed.")

Logistic Regression tuning completed.


c:\Users\mvans\OneDrive\Desktop\Healytics\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [27]:
print("Best parameters:")
print(lr_grid_search.best_params_)

print("\nBest cross-validation ROC-AUC:")
print(round(lr_grid_search.best_score_, 4))

best_lr_model = lr_grid_search.best_estimator_

print("\nBest Logistic Regression pipeline created successfully.")

Best parameters:
{'model__C': 1, 'model__penalty': 'l2'}

Best cross-validation ROC-AUC:
0.843

Best Logistic Regression pipeline created successfully.


## 9. Tuned Model Cross-Validation Comparison

The tuned models are compared using their best mean cross-validation ROC-AUC.

These scores were obtained using only the training dataset.

The test dataset has not been used for hyperparameter selection.


In [28]:
tuned_cv_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Best CV ROC-AUC": [
        lr_grid_search.best_score_,
        rf_grid_search.best_score_,
        gb_grid_search.best_score_
    ]
})

tuned_cv_results.round(4)

,Model,Best CV ROC-AUC
0,Logistic Regression,0.8430
1,Random Forest,0.8375
2,Gradient Boosting,0.8297


In [29]:
tuned_cv_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Best CV ROC-AUC": [
        lr_grid_search.best_score_,
        rf_grid_search.best_score_,
        gb_grid_search.best_score_
    ]
})

tuned_cv_results.round(4)

,Model,Best CV ROC-AUC
0,Logistic Regression,0.8430
1,Random Forest,0.8375
2,Gradient Boosting,0.8297


## Interpretation

The tuned models achieved different cross-validation ROC-AUC scores.

The cross-validation results are used to identify promising tuned candidates,
but they are not treated as the final test performance.

The final comparison will be performed on the previously untouched test set.

## 10. Final Test Evaluation of Tuned Models

The best tuned version of each candidate model is evaluated on the untouched
test dataset.

The test set is used only at this stage to estimate performance on unseen
data.

The same metrics used during baseline evaluation are calculated:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

In [31]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [32]:
tuned_models = {
    "Tuned Logistic Regression": best_lr_model,
    "Tuned Random Forest": best_rf_model,
    "Tuned Gradient Boosting": best_gb_model
}

tuned_test_results = []

for name, model in tuned_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    tuned_test_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

tuned_test_results_df = pd.DataFrame(tuned_test_results)

tuned_test_results_df.round(4)

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Tuned Logistic Regression,0.7078,0.6000,0.5000,0.5455,0.8130
1,Tuned Random Forest,0.7403,0.6667,0.5185,0.5833,0.8144
2,Tuned Gradient Boosting,0.7208,0.6222,0.5185,0.5657,0.8148


## 11. Hyperparameter Tuning Findings

Hyperparameter tuning was performed using 5-fold cross-validation on the
training dataset only.

The best cross-validation ROC-AUC scores were:

- Logistic Regression: 0.8430
- Random Forest: 0.8375
- Gradient Boosting: 0.8297

The tuned models were then evaluated once on the previously untouched test
dataset.

The tuned test-set ROC-AUC values were:

- Logistic Regression: 0.8130
- Random Forest: 0.8144
- Gradient Boosting: 0.8148

The tuned models did not improve ROC-AUC over all corresponding baseline
models on this particular test split.

This demonstrates that cross-validation performance and test-set performance
can differ. The test set is therefore retained as an independent final
evaluation set and is not used for further hyperparameter selection.

The cross-validation results will be considered when selecting the final
candidate model, together with the complete evaluation metrics and the
requirements of the diabetes risk prediction task.

## 12. Baseline vs Tuned Model Comparison

The baseline models are retrained using the same training/test split and
evaluation procedure used during the baseline evaluation.

Their test-set performance is then compared with the tuned models.

This comparison is descriptive and is used to document the effect of
hyperparameter tuning.

In [34]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [35]:
baseline_models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", GradientBoostingClassifier(
            random_state=42
        ))
    ])
}

In [36]:
baseline_results = []

for name, model in baseline_models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    baseline_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Version": "Baseline"
    })

baseline_comparison = pd.DataFrame(baseline_results)

baseline_comparison.round(4)

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Version
0,Logistic Regression,0.7078,0.6000,0.5000,0.5455,0.8130,Baseline
1,Random Forest,0.7727,0.7021,0.6111,0.6535,0.8181,Baseline
2,Gradient Boosting,0.7532,0.6818,0.5556,0.6122,0.8302,Baseline


In [37]:
tuned_comparison = tuned_test_results_df.copy()
tuned_comparison["Version"] = "Tuned"

comparison_df = pd.concat(
    [baseline_comparison, tuned_comparison],
    ignore_index=True
)

comparison_df = comparison_df[
    [
        "Version",
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "ROC-AUC"
    ]
]

comparison_df.round(4)

,Version,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Baseline,Logistic Regression,0.7078,0.6000,0.5000,0.5455,0.8130
1,Baseline,Random Forest,0.7727,0.7021,0.6111,0.6535,0.8181
2,Baseline,Gradient Boosting,0.7532,0.6818,0.5556,0.6122,0.8302
3,Tuned,Tuned Logistic Regression,0.7078,0.6000,0.5000,0.5455,0.8130
4,Tuned,Tuned Random Forest,0.7403,0.6667,0.5185,0.5833,0.8144
5,Tuned,Tuned Gradient Boosting,0.7208,0.6222,0.5185,0.5657,0.8148


## 13. M7 Conclusion

Hyperparameter tuning was performed using 5-fold cross-validation on the
training dataset.

The best cross-validation ROC-AUC scores were:

- Logistic Regression: 0.8430
- Random Forest: 0.8375
- Gradient Boosting: 0.8297

The tuned models were subsequently evaluated on the untouched test set.

The test-set results did not show an improvement in ROC-AUC compared with the
corresponding baseline models on this particular split.

This demonstrates that improved cross-validation performance does not
necessarily translate into improved performance on an unseen test set.

The test set was not used during hyperparameter selection and remains a valid
independent evaluation set.

The tuned pipelines are retained for further model-selection analysis.